In [1]:
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from urllib.parse import urljoin
import re
from translate import Translator
import pandas as pd
import numpy as np

In [2]:
def convert_image(image_path, image_format):
    image = Image.open(image_path)
    image = image.convert('RGB')
    image.save(image_path, image_format)

def download_image(url, save_path, image_format):
    response = requests.get(url)
    if response.status_code == 200:
        with open(save_path, 'wb') as file:
            file.write(response.content)
        # end
        
        if url.split('.')[-1] != 'jpg':
            convert_image(save_path, image_format)
        # end
        
        print(f"Image downloaded and converted to {image_format} successfully: {save_path}.")
    else:
        print("Failed to download the image.")

def resize_image(url, output_path, new_width):
    # Download the image from the URL
    response = requests.get(url, stream=True)
    response.raise_for_status()

    # Open the downloaded image using PIL
    image = Image.open(response.raw)
    
    # Calculate the new height to maintain aspect ratio
    orig_width, orig_height = image.size
    new_height = int(new_width * orig_height / orig_width)
    
    #print(image.size)
    
    # Resize the image
    #image = image.resize((new_width, new_height))

    # Save the resized image
    image.save(output_path, 'JPEG')

    # Close the image
    image.close()

In [59]:
#translator = Translator(to_lang="en", from_lang="de", service="yandex")

data_dir = '../../../Data/Papyri Databases/PSI/'
img_dir  = '../../../Data/Papyri Databases/PSI/PSI_images/'

data_file = 'PSI_links.csv'

In [61]:
base_url = 'http://www.psi-online.it/documents'

"""
df = pd.DataFrame(columns=['url', 'edition', 'inventory', 'typology', 'storage',
                           'origin', 'material', 'library_typology', 'content_r',
                           'content_v', 'dating', 'date', 'num_frags', 'dimensions', 
                           'content', 'note', 'further_info', 'trismegistos',
                           'LDB_ext' ])
"""

df = pd.DataFrame(columns=['url'])

links = []
#for i in range(0,10):
for i in range(0,178):
    #print(i)
    
    # check page url exists
    page_id = '?page=' + str(i)
    page_url = base_url + page_id
    
    response = requests.get(page_url)
    
    if response.status_code != 200:
        #print("Invalid: " + page_url)
        continue
    # end
    
    #print("Valid:   " + page_url)
    
    # find doc links on page
    try:
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')

        data = soup.body
        data = data.find('div', class_='container')
        data = data.find('div', id='content')
        data = data.find('section')
        data = data.find('table')
        trs = data.find_all('tr')[1:]
        
        for i in range(len(trs)):
            x = trs[i].find('td')
            x = x.find('a')['href']
            x = x.split('/')[-1]
            
            links.append( base_url + '/' + x )
        # end
    except:
        #print("Failed to parse HTML")
        continue
    # end
# end

df.url = links

df.head()

df.to_csv(data_dir + data_file, index=False)